In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('data.csv')
df = df.drop_duplicates()
df['chronic_conditions'] = df['chronic_conditions'].fillna('None')
df['alcohol_use'] = df['alcohol_use'].fillna('None')

X = df.drop(['readmission_risk', 'patient_id'], axis=1) 
y = df['readmission_risk']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def neural_network_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(input_shape,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(3, activation='softmax')
    ])
    # FIX: Correct spelling of 'sparse'
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def Drop_Out_Model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu',input_shape=(input_shape,)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(3,activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

simple_model = neural_network_model(X_train_scaled.shape[1])

simple_history = simple_model.fit(
    X_train_scaled, y_train, 
    epochs=100, 
    batch_size=32, 
    validation_split=0.2, 
    callbacks=[early_stop], 
    verbose=1
)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(simple_history.history['loss'], label='Training Loss')
plt.plot(simple_history.history['val_loss'], label='Validation Loss')
plt.title('Simple Model Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(simple_history.history['accuracy'], label='Training Acc')
plt.plot(simple_history.history['val_accuracy'], label='Validation Acc')
plt.title('Simple Model Accuracy')
plt.legend()
plt.show()

loss, accuracy=simple_model.evaluate(X_test_scaled, y_test)
print('Accuracy of Simple Neural Network Model is: ', accuracy)

dropout_model = Drop_Out_Model(X_train_scaled.shape[1])

dropout_history = dropout_model.fit(
    X_train_scaled, y_train, 
    epochs=100, 
    batch_size=32, 
    validation_split=0.2, 
    callbacks=[early_stop], 
    verbose=1
)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(dropout_history.history['loss'], label='Training Loss')
plt.plot(dropout_history.history['val_loss'], label='Validation Loss')
plt.title('Drop Out Model Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(dropout_history.history['accuracy'], label='Training Acc')
plt.plot(dropout_history.history['val_accuracy'], label='Validation Acc')
plt.title('Drop Out Model Accuracy')
plt.legend()
plt.show()

d_loss, d_accuracy=dropout_model.evaluate(X_test_scaled, y_test)
print('Accuracy of Drop Out Neural Network Model is: ', d_accuracy)

def K_Fold_Validation(X_data, y_data, model_type='simple'):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, test_idx in kf.split(X_data):
        X_train_kf, X_test_kf = X_data[train_idx], X_data[test_idx]
        y_train_kf, y_test_kf = y_data[train_idx], y_data[test_idx]


    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_data.shape[1],)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer='adam', 
        loss='sparse_categorical_crossentropy', 
        metrics=['accuracy']
    )

    model.fit(
        X_train_kf, y_train_kf, 
        epochs = 100, 
        batch_size = 32, 
        validation_data = (X_test_kf, y_test_kf),
        callbacks = [early_stop], 
        verbose = 0
    )

    loss, accuracy = model.evaluate(X_test_kf, y_test_kf, verbose=0)
    scores.append(accuracy)
    return scores, np.mean(scores)

X_all_scaled = scaler.fit_transform(X_encoded)
simple_scores, mean_acc_simple = K_Fold_Validation(X_all_scaled, y_encoded, model_type='simple')
dropout_scores, mean_acc_dropout = K_Fold_Validation(X_all_scaled, y_encoded, model_type='dropout')

print("Simple Model Average Accuracy:", mean_acc_simple)
print("Dropout Model Average Accuracy:", mean_acc_dropout)      

rf_model = RandomForestClassifier(random_state=42)
rf_cv_scores = cross_val_score(rf_model, X_encoded, y_encoded, cv=5)
rf_model.fit(X_train, y_train)
rf_test_acc = rf_model.score(X_test, y_test)

print(f"Random Forest 5-Fold CV Accuracy: {rf_cv_scores.mean():.4f}")
print(f"Random Forest Test Accuracy: {rf_test_acc:.4f}")

# Both the TensorFlow Neural Networks and the Scikit-learn Random Forest yielded similar results. In this specific case, the TensorFlow Dropout Model performed slightly better in Cross-Validation but the difference is marginal